In [7]:
import re

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool

In [43]:
@tool
def customer_lookup(query : str) -> str:
    """Look up customer information"""
    return f"Customer record found for query: {query}"

# Create agent with PII Middleware
agent = create_agent(
    model = "mistral-small-2506",
    tools = [customer_lookup],
    middleware = [

        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_input = True
        ),

        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            detector=r"\b(?:\d{4}[- ]?){3}\d{4}\b",
            strategy = "mask",
            apply_to_input=True
        ),

        #Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector = r"sk-[a-zA-Z0-9]{32}",
            strategy = "block",
            apply_to_input = True
        ),
    ],
)

print("Agent with PII Middleware created successfully!")

Agent with PII Middleware created successfully!


In [44]:
#TEST PII REDACTION

result = agent.invoke({
    "messages" : [{ 
        "role" : "user",
        "content" :(
            "My email is john.doe@example.com"
            "and my card is 4111-1111-2222-3333. Can you help me?"
        )
    }]
})

print("Agent response")
print(result["messages"][0].content)
print(result["messages"][-1].content)

Agent response
My email is [REDACTED_EMAIL] my card is ****-****-****-3333. Can you help me?
I'm really sorry, but I'm unable to assist with that specific request. However, I'm here to help with any other questions or concerns you might have. Please feel free to ask!


In [45]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] my card is ****-****-****-3333. Can you help me?', additional_kwargs={}, response_metadata={}, id='d62e38da-6f43-4b0b-a5f3-3668b66a9546'),
  AIMessage(content="I'm really sorry, but I'm unable to assist with that specific request. However, I'm here to help with any other questions or concerns you might have. Please feel free to ask!", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 89, 'total_tokens': 129, 'completion_tokens': 40, 'prompt_tokens_details': {'cached_tokens': 64}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019f65dc-a7ba-72c3-9435-0327095f4fcc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 89, 'output_tokens': 40, 'total_tokens': 129})]}

In [46]:
#Test API key blocking

try:
    result = agent.invoke({
        "messages": [{
            "role":"user",
            "content": "here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"Blocked as Expected: {e}") 

Blocked as Expected: Detected 1 instance(s) of api_key in text content
